# Train100K scale survival — matched crop1.5

Exactly **C_100K_crop15**, **P3_SF_100K_crop15**, **R7_SC_100K_crop15**. Notebook09 supplies the frozen scale infrastructure; notebooks13/14 supply established SF/SC mechanisms. Existing notebooks/resources remain unchanged.

1. Edit resource/image/model paths and **SCALE_MANIFEST_ROOT** pointing to existing frozen `scale100k_train.csv` / `scale15k_val.csv`. No sampling or detector runs.
2. `RUN_FILTER=None` runs all; `['C_100K_crop15']`, `['P3_SF_100K_crop15']`, `['R7_SC_100K_crop15']` or any valid subset supports independent sessions. `RESUME=True` restores each run separately.
3. Run setup, fairness and lightweight checks, then review pending report/export. All three initial-state SHAs and manifest identities must match before training, even with a filtered session.
4. Final runner starts expensive GPU training. Completed run ZIP is saved immediately before moving on.

**C is eligible from epoch1**; P3/R7 warmup epoch1 is excluded from best/patience. All Train/Val15K/full CelebA Test/LCC crops are **1.5**. P3_SF sigma=**.0555555556**; R7_SC sigma=**.10**, with the same scaled centers. Best/threshold use only Val15K. Full Test and LCC inference follow source freeze.

**New shared phase-aware patience:** start counting non-improvement at epoch15, patience3, earliest possible stop afterepoch17, max20. Validation/best selection timing is unchanged. This differs from older historical100K training protocols.

Scale-survival gate uses **only Combined-AUC delta versus matched C100K**: ≥.005 strong, >0 weak, otherwise no survival. No automatic tuning or follow-up experiment.


In [ ]:
# Edit Kaggle paths here. Mount existing frozen 100K/15K manifests separately.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
SCALE_MANIFEST_ROOT='/kaggle/input/datasets/maithanhphuong/100k-output/binary_crossdomain_scale100k_C_P3_Q3_v1/manifests'
OUTPUT_ROOT='/kaggle/working/17_scale100k_C_P3SF_R7SC_crop15'
RUN_FILTER=None  # Or any valid subset of the three exact run IDs below.
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
PATIENCE_START_EPOCH=15
EARLY_STOP_PATIENCE=3
WARMUP_EPOCHS=1
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=1.5
TRAIN_CROP_FACTOR=1.5
EVAL_CROP_FACTOR=1.5
LOG_EVERY_BATCHES=25
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12

SCALED_CENTERS=[c*(1.5/2.7) for c in (.15,.45,.75)]
P3_SF_SIGMA=.10*(1.5/2.7)
R7_SC_SIGMA=.10
RUNS={
 'C_100K_crop15':{'method':'CLEAN','warmup_epochs':0,'crop_factor':1.5,'centers':None,'sigma':None,'clean_ce_weight':1.,'spectral_ce_weight':0.},
 'P3_SF_100K_crop15':{'method':'WORST','warmup_epochs':1,'crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':P3_SF_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25},
 'R7_SC_100K_crop15':{'method':'HARMFUL','warmup_epochs':1,'crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':R7_SC_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25},
}
def select_runs(run_filter):
    if run_filter is not None and (not isinstance(run_filter,(list,tuple)) or not run_filter or len(set(run_filter))!=len(run_filter) or set(run_filter)-set(RUNS)):
        raise ValueError('RUN_FILTER must be None or a non-empty unique subset of C_100K_crop15, P3_SF_100K_crop15, R7_SC_100K_crop15')
    return [name for name in RUNS if run_filter is None or name in run_filter]


In [ ]:
PROTOCOL_TEXT='# Matched crop1.5 Train100K scale-survival protocol v1\n\nQuestion: do established micro C/P3_SF/R7_SC mechanisms retain or strengthen their cross-domain gains at frozen Train100K/Val15K scale? Exactly C_100K_crop15, P3_SF_100K_crop15, R7_SC_100K_crop15. No new method/search/seed. Infrastructure from notebook09; P3_SF from13/15; R7_SC from14; matched crop from12. Prior notebooks/resources remain unchanged.\n\nRead scale100k_train.csv and scale15k_val.csv directly from editable SCALE_MANIFEST_ROOT, preserving existing notebook09 eligibility/label checks and source order. Copy exact bytes into artifacts and record SHA256. No split regeneration, resampling or detector. Data43/train100. Before training, initialize all three with the same notebook09 official PAD-pretrained policy and require identical state SHA; per-run initialization repeats that policy and checks the same SHA. Validate each config uses identical manifest hashes and train/val/test/LCC crop1.5.\n\nArchitecture unchanged: MiniFASNetV2 128D, PAD128->2, spoof11/light5/attrs40. BGR float32[0,1]3x80x80, no normalization. Exact current CropImage-compatible helper, bbox jitter and worker augmentation/data-order RNG from09/04. Spectral gain independent torch generator seedTRAIN_SEED+11000. Shared eval architecture is clean-only; spectral logic exists only during training.\n\nFrozen batch256 AMP, SGD(.005,.9,5e-4), max20, MultiStepLR[6,14]gamma.2 stepped after each epoch, phase-aware PATIENCE_START_EPOCH15/EARLY_STOP_PATIENCE3, earliest possible stop afterepoch17, METRIC_TIE_EPS1e-12. Val15K minACER selects best, higherAUC tie-break. Source-only threshold/checkpoint/patience. New shared phase-aware patience changes only when non-improvement counts: beforeepoch15 counter stayszero; fromepoch15 improvement resetszero and non-improvement increments; stop atcounter3, max20. Best/validation timing unchanged. This experiment is not training-protocol-identical to older historical scale100K runs. Historical notebook04 explicitly sets C warmup0/eligible epoch1 and P3 warmup1/excluded best/patience; notebook09 R7 also excludes epoch1. These semantics are preserved. Full Test/LCC cannot influence training or source decisions.\n\nC: clean CE+aux; no masks or selection. P3_SF: centers[.15,.45,.75]*(1.5/2.7), sigma.10*(1.5/2.7), .75 clean+.25 mean worst CE+aux across all examples. R7_SC: same centers, sigma.10 original bandwidth, .75 clean+.25 harmful CE+aux. GainUniform(.65,.90) one/sample shared bands/channels, preserve DC/phase/clipping and selection chunks32. P3/R7 eval/no_grad clean+LOW/MID/HIGH signed margin selection and restore prior mode; only selected worst gets spectral gradient. R7 harmful=(worst_margin<clean_margin).detach, independent of misclassification; CE denominator harmful_count, differentiable logits.sum()*0 ifnone. Active clean coefficient stays.75 even ifnoharmful. Epoch1 spectral methods clean CE+aux only.\n\nAux exactly .1 spoof CE+.1 lighting CE+1.0 Real-only attributes BCE, clean-view only. No severity/dynamic/stochastic/margin-consistency/new losses or geometry tuning.\n\nReload eligible best, infer Val15Kcrop1.5 and recompute/freeze source minACER threshold in evaluation_freeze.json. Only then infer full CelebA Testcrop1.5, followed by LCC physical train/dev/evalcrop1.5. Concatenate candidate prediction tables including localization failures and recompute pooled Combined metrics over valid scores, never average split metrics. Preserve candidate/scored/coverage; rawLCC8299/2948/7580/18827, Test67170.\n\nEach runs/<run_id>/ has config, init audit, best, last_state, histories, source freeze, completion, metrics/ and predictions/. P3/R7 spectral_diagnostics.csv contains only active epochs as in09. Preserve model/optimizer/scheduler/scaler/epoch/best/patience/history/global/gain/bbox RNG. Worker/epoch data-order/augmentation/bbox streams regenerate exactly at epoch boundaries. Notebook09 stale duplicate restore_rng override is removed in this new notebook only; keep correct two-argument restore including bbox state. Explicit initial SHA in resume state and config/run checks reject cross-run/stale state. OOM onbatch256 stops loudly; no automatic recipe change.\n\nLogs include training/inference batch progress. C loss/aux diagnostics retained; P3 clean/worst/total, margins/drop/flip/gain, selectedband counts/fractions; R7 adds harmfulcount/fraction/CE. Epoch harmfulcount is summed; harmful_ce uses pooled harmful denominator; weighted batch mean is retained separately. These diagnostics do not change loss or decisions.\n\nVerified historical micro CombinedAUC reporting references supplied by user: C.830642, P3_SF.837525, R7_SC.832523; deltas.006883/.001881. Read only after training definitions, never in sampling/RNG/config/selection. Primary scale endpoint measured CombinedAUC. P3/R7 delta versus matched C100K >=.005 STRONG_SCALE_SURVIVAL, >0 WEAK_SCALE_SURVIVAL, otherwise NO_SCALE_SURVIVAL. Missing baseline meanspending. No secondary override. Delta percentage points and descriptive retention delta/micro_delta reported; R7 denominator small and not used for gating/ranking.\n\nAfter all runs, descriptive highest CombinedAUC gives best100k method, ties at six-decimal precision reported without selecting a winner. P3SFvsR7SC delta reported. Recommendations only: retain strongest strong P3_SF orR7_SC; bothstrong viable, rankdescriptively, noautomaticcombination; weak directional modest benefit warrants different mechanism; neither meanspivot beforemore100K compute. No automatic follow-up or local weight/geometry tuning.\n\nImmediately archive each completed run to /kaggle/working/<run_id>.zip before next run and before report generation; include protocol/audits/frozen manifests. Refresh scale100k_C_P3SF_R7SC_comparison.csv, scale100k_C_P3SF_R7SC_decision.json, scale100k_C_P3SF_R7SC_report.md and /kaggle/working/17_scale100k_C_P3SF_R7SC_crop15.zip; FileLinks. Independent filtered Kaggle sessions supported, RUN_FILTERNone or any nonempty unique subset; RESUMETrue.\n\nValidation is offline syntax/cleared outputs, inherited-helper comparisons, exact objective/backward synthetic tests against13/14, zero-harmful/BN/warmup/RNG tests, manifest/init equality contracts including rejection cases, freeze ordering, pooled prediction/coverage checks, isolated resume/ZIP/report fixtures. No expensive local training or actual dataset inference. Verify editable resource/image/model paths, all source/Test/LCC images and T4batch256memory onKaggle.\n\nLimitations: one seed, repeatedly used LCC external-development benchmark, engineering screen not significance; no statistical superiority or final independent-test generalization claim. Retention is descriptive, smallR7denominator unstable. Multi-seed/additional independent validation future work. During notebook creation no100K result has been measured.\n'

import hashlib, time, shutil, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

_BBOX_RNG=random.Random(TRAIN_SEED+500000)
AUG_SEED=TRAIN_SEED
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
if RUN_FILTER is not None and (not isinstance(RUN_FILTER,(list,tuple)) or not RUN_FILTER or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):
    raise ValueError('RUN_FILTER contains an unknown or duplicate run ID')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen scale100K recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,Path(SCALE_MANIFEST_ROOT)/'scale100k_train.csv',Path(SCALE_MANIFEST_ROOT)/'scale15k_val.csv',
             root/'manifests/crossdomain_celeba_source_dev_full.csv',root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(centers,sigma,size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(centers,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*sigma**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain']);_BBOX_RNG.setstate(state['bbox'])
def bbox_jitter(rec):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if _BBOX_RNG.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=_BBOX_RNG.uniform(.95,1.05)
        cx+=_BBOX_RNG.uniform(-.05,.05)*base;cy+=_BBOX_RNG.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]


def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=pd.read_csv(Path(SCALE_MANIFEST_ROOT)/'scale100k_train.csv',keep_default_na=False)
val=pd.read_csv(Path(SCALE_MANIFEST_ROOT)/'scale15k_val.csv',keep_default_na=False)
test=load_frame('crossdomain_celeba_source_dev_full')
if (len(train),len(val),len(test))!=(100000,15000,67170):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if set(frame.sampling_seed)!={DATA_SEED}:raise RuntimeError('Frozen data seed differs')
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen scale split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen scale100K batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'manifests').mkdir(exist_ok=True)
for filename in ('scale100k_train.csv','scale15k_val.csv'):
    shutil.copyfile(Path(SCALE_MANIFEST_ROOT)/filename,out/'manifests'/filename)
(out/'SCALE100K_C_P3SF_R7SC_CROP15_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
MANIFEST_HASHES={f:sha(Path(SCALE_MANIFEST_ROOT)/f) for f in ('scale100k_train.csv','scale15k_val.csv')}
for filename,expected_hash in MANIFEST_HASHES.items():
    if sha(out/'manifests'/filename)!=expected_hash:raise RuntimeError('Copied frozen manifest differs')
write_json(out/'manifest_sha256.json',MANIFEST_HASHES)
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()})

def init_model(spec):
    seed_all(TRAIN_SEED);model=PADNet(spec['copied']);audit={'initialization':spec['init'],'checkpoint_sha256':sha(pretrained),'source_sha256':sha(source)}
    if spec['init']=='pretrained':
        raw=torch.load(pretrained,map_location='cpu',weights_only=True)
        if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):raise RuntimeError('Official checkpoint format mismatch')
        stripped=OrderedDict((k[7:],v) for k,v in raw.items())
        if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official checkpoint architecture mismatch')
        target=model.trunk.state_dict();compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
        bad=[k for k,v in compatible.items() if k not in target or tuple(v.shape)!=tuple(target[k].shape)]
        absent=[k for k in target if k not in compatible and k!='prob.weight']
        if bad or absent:raise RuntimeError(f'Pretrained key mismatch: {bad[:5]} {absent[:5]}')
        missing,unexpected=model.trunk.load_state_dict(compatible,strict=False)
        if unexpected or (spec['copied'] and missing) or (not spec['copied'] and set(missing)!={'prob.weight'}):raise RuntimeError('Pretrained load mismatch')
        # Reinitialize historical classifier too; copied heads were initialized by seed 100.
        if not spec['copied']:model.trunk.prob.reset_parameters()
        audit.update({'loaded_keys':sorted(compatible),'skipped_keys':['prob.weight'],'missing_keys':list(missing),'unexpected_keys':list(unexpected)})
    else:audit.update({'loaded_keys':[],'skipped_keys':[],'missing_keys':[],'unexpected_keys':[]})
    audit['initial_state_sha256']=state_sha(model.state_dict())
    with torch.no_grad():
        model.eval();assert model(torch.zeros(2,3,80,80)).shape==(2,2 if spec['copied'] else 3)
    return model,audit

INITIAL_AUDITS={}
for name in RUNS:
    initial_model,initial_audit=init_model({'init':'pretrained','copied':True})
    INITIAL_AUDITS[name]=initial_audit
    del initial_model
INITIAL_HASHES={name:audit['initial_state_sha256'] for name,audit in INITIAL_AUDITS.items()}
if len(set(INITIAL_HASHES.values()))!=1:raise RuntimeError('Initial-state SHA differs across all three methods')
INITIAL_STATE_SHA=next(iter(INITIAL_HASHES.values()))
write_json(out/'initialization_fairness.json',INITIAL_AUDITS)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,copied=False):self.rows=frame.reset_index(drop=True);self.training=training;self.copied=copied
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        patch=mini_crop_bgr(image,bbox_jitter(rec) if self.training else rec['bbox_xyxy'])
        if self.training:patch=copied_aug_bgr(patch) if self.copied else mild_aug(patch)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key

class LCCData(Dataset):
    def __init__(self,split):
        self.split=split;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,batch=BATCH_SIZE,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(wid):
        global _BBOX_RNG
        s=TRAIN_SEED+epoch*1000+wid;random.seed(s);np.random.seed(s);torch.manual_seed(s)
        _BBOX_RNG=random.Random(s+500000)
        if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(s)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,drop_last=False,generator=gen,worker_init_fn=worker_init)

def infer(model,dl,lcc_mode=False):
    model.eval();keys=[];ys=[];zs=[];started=time.monotonic()
    print('[Inference] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc_mode:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):print(f'[Inference] batch {step}/{len(dl)}, elapsed={time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable evaluation samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))


def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active):
    method=spec['method'];n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_worst=None
    spectral_active=active and method!='CLEAN'
    if spectral_active:
        selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;worst_ce=clean_ce*0;harmful_count=0;spectral_logits=None;harmful_ce_sum=0.
        if spectral_active:
            spectral_logits=model(selected);ce_each=F.cross_entropy(spectral_logits,binary,reduction='none');worst_ce=ce_each.mean()
            if method=='HARMFUL':
                harmful=(select_worst<select_clean).detach();harmful_count=int(harmful.sum())
                harmful_ce_sum=float(ce_each.detach()[harmful].sum())
                spectral_ce=(ce_each*harmful).sum()/harmful_count if harmful_count else spectral_logits.sum()*0
            else:spectral_ce=worst_ce
        loss=.75*clean_ce+.25*spectral_ce+aux_total if spectral_active else clean_ce+aux_total
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary).mean()
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
          'spectral_ce':float(spectral_ce.detach()),'worst_ce':float(worst_ce.detach()),'aux_loss':float(aux_total.detach()),
          'spoof_type_ce':float(ls.detach()),'lighting_ce':float(ll.detach()),'attributes_real_bce':float(la.detach()),
          'clean_aligned_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
          'clean_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
          'train_clean_aligned_margin':float(cm),'worst_margin':float(select_worst.mean()) if select_worst is not None else 0.,
          'margin_drop':float((select_clean-select_worst).mean()) if select_worst is not None else 0.,
          'spectral_flip_rate':flips/n,'harmful_count':harmful_count,'harmful_fraction':harmful_count/n,
          'harmful_ce':float(spectral_ce.detach()) if method=='HARMFUL' and spectral_active else 0.,
          'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts,harmful_ce_sum

def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state(),'bbox':_BBOX_RNG.getstate()}

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'scale100k_train.csv',
      'val_manifest':'scale15k_val.csv','scale_manifest_root':SCALE_MANIFEST_ROOT,'manifest_sha256':MANIFEST_HASHES,
      'train_crop_factor':1.5,'val_crop_factor':1.5,'test_crop_factor':1.5,'lcc_crop_factor':1.5,'data_seed':DATA_SEED,
      'train_seed':TRAIN_SEED,'initial_state_sha256':INITIAL_STATE_SHA,
      'base_order':'torch Generator(TRAIN_SEED+epoch)','worker_seed':'TRAIN_SEED+epoch*1000+worker_id',
      'bbox_rng_seed':'worker_seed+500000','gain_rng_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),
      'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,'num_workers':NUM_WORKERS,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':spec['warmup_epochs'],'patience_start_epoch':PATIENCE_START_EPOCH,'early_stop_patience':EARLY_STOP_PATIENCE,
      'spectral':None if spec['method']=='CLEAN' else {'centers':spec['centers'],'sigma':spec['sigma'],'gain':[.65,.90],'preserve_dc':CSMR_PRESERVE_DC,'selection_chunk_size':SELECTION_CHUNK_SIZE},
      'input':'80x80 BGR float [0,1] crop1.5','augmentation_api':AUGMENTATION_API,
      'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked):
    scored_predictions=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z=infer(model,loader(LCCData(split)),True)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/'metrics'/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'metrics'/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)

def evaluate_celeba_test(model,folder,locked):
    scored=test.loc[test.usable_for_pad.astype(bool)]
    keys,y,z=infer(model,loader(CelebaData(scored)))
    if keys!=scored.sample_key.astype(str).tolist():raise RuntimeError('CelebA Test order differs')
    write_json(folder/'metrics'/'test_metrics.json',metrics(y,pad_score(z),locked,len(test)))
    prediction_frame(test,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'test_predictions.csv',index=False)

def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True);(folder/'metrics').mkdir(exist_ok=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/'metrics'/(k+'_metrics.json') for k in ('val','test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping')
        return
    model,audit=init_model({'init':'pretrained','copied':True})
    if audit['initial_state_sha256']!=INITIAL_STATE_SHA:raise RuntimeError('Initial state differs')
    write_json(folder/'initialization_audit.json',audit)
    model=model.to(device)
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state.get('initial_state_sha256')!=INITIAL_STATE_SHA:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume best payload differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start)
    val_dl=loader(CelebaData(val))
    masks=radial_masks(spec['centers'],spec['sigma']) if spec['method']!='CLEAN' else None
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            expected_lr=.005 if epoch<=6 else (.001 if epoch<=14 else .0002)
            if not math.isclose(optimizer.param_groups[0]['lr'],expected_lr,rel_tol=1e-9):raise RuntimeError('Scheduler LR differs')
            warmup=epoch<=spec['warmup_epochs']
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;harmful_count_total=0;harmful_ce_sum_total=0.;epoch_started=time.monotonic()
            train_ds=CelebaData(train,training=True,copied=True)
            train_dl=loader(train_ds,shuffle=True,epoch=epoch)
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, warmup={warmup}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_) in enumerate(train_dl,1):
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands,harmful_ce_sum=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                harmful_count_total+=scalars['harmful_count'];harmful_ce_sum_total+=harmful_ce_sum
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-epoch_started:.1f}s',flush=True)
            keys,y,z=infer(model,val_dl)
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                no_improve=advance_patience(no_improve,improved,epoch,active)
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            if spec['method']=='HARMFUL':
                row['harmful_ce_batch_mean']=row['harmful_ce']
                row['harmful_ce']=harmful_ce_sum_total/harmful_count_total if harmful_count_total else 0.
            if active and spec['method']!='CLEAN':diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or patience_stop_reached(epoch,no_improve)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'initial_state_sha256':INITIAL_STATE_SHA,'rng':snapshot_rng(gain_rng),'best_checkpoint':best_checkpoint,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            if spec['method']!='CLEAN':pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<=spec['warmup_epochs']:
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z=infer(model,val_dl)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'metrics'/'val_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'val_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val_metrics':vm,'manifest_sha256':MANIFEST_HASHES,
      'train_crop_factor':1.5,'val_crop_factor':1.5,'test_crop_factor':1.5,'lcc_crop_factor':1.5}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_celeba_test(model,folder,locked)
    evaluate_lcc(model,folder,locked)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'initial_state_sha256':INITIAL_STATE_SHA})
    del model
    torch.cuda.empty_cache()

def assert_fairness_contract(configs,initial_hashes):
    if set(configs)!=set(RUNS) or set(initial_hashes)!=set(RUNS):raise RuntimeError('Fairness audit requires all three runs')
    if len(set(initial_hashes.values()))!=1 or not next(iter(initial_hashes.values())):raise RuntimeError('Initial-state SHA differs across methods')
    expected={'scale100k_train.csv','scale15k_val.csv'}
    for cfg in configs.values():
        if set(cfg['manifest_sha256'])!=expected or cfg['manifest_sha256']!=MANIFEST_HASHES:raise RuntimeError('Frozen manifest SHA differs across methods')
        if cfg['initial_state_sha256']!=initial_hashes[cfg['run_id']]:raise RuntimeError('Config/init SHA differs')
        if any(cfg[k]!=1.5 for k in ('train_crop_factor','val_crop_factor','test_crop_factor','lcc_crop_factor')):raise RuntimeError('Matched crop1.5 required')
    c=RUNS['C_100K_crop15'];p=RUNS['P3_SF_100K_crop15'];r=RUNS['R7_SC_100K_crop15']
    if c['method']!='CLEAN' or c['warmup_epochs']!=0 or c['centers'] is not None or c['sigma'] is not None or (c['clean_ce_weight'],c['spectral_ce_weight'])!=(1.,0.):raise RuntimeError('Historical C mechanism drift')
    for spec in (p,r):
        if spec['centers']!=SCALED_CENTERS or (spec['clean_ce_weight'],spec['spectral_ce_weight'],spec['warmup_epochs'])!=(.75,.25,1):raise RuntimeError('Historical spectral mechanism drift')
    if p['method']!='WORST' or p['sigma']!=.10*(1.5/2.7) or r['method']!='HARMFUL' or r['sigma']!=.10:raise RuntimeError('SF/SC geometry mismatch')
    return True

def advance_patience(patience_counter,improved,epoch,eligible):
    # Checkpoint eligibility and improvement are determined before this rule.
    if improved or epoch<PATIENCE_START_EPOCH:return 0
    if not eligible:return patience_counter
    return patience_counter+1

def patience_stop_reached(epoch,patience_counter):
    return epoch>=PATIENCE_START_EPOCH and patience_counter>=EARLY_STOP_PATIENCE

def phase_patience_sanity_checks():
    for spec in RUNS.values():
        counter=0;stopped=None
        for epoch in range(1,21):
            eligible=epoch>spec['warmup_epochs']
            counter=advance_patience(counter,False,epoch,eligible)
            if epoch<15:assert counter==0
            if patience_stop_reached(epoch,counter):stopped=epoch;break
        assert stopped==17
    counter=advance_patience(2,True,16,True)
    assert counter==0
    for epoch in (17,18,19):counter=advance_patience(counter,False,epoch,True)
    assert counter==3 and patience_stop_reached(19,counter)
    assert advance_patience(0,True,10,True)==0
    assert not patience_stop_reached(14,3)
    # Resuming committed epoch14 vs uninterrupted final-phase counts gives identical results.
    first=advance_patience(0,False,15,True);resumed=advance_patience(first,False,16,True)
    assert resumed==2 and advance_patience(resumed,False,17,True)==3
    return {'patience_start_epoch':15,'early_stop_patience':3,'earliest_possible_stop_epoch':17,
            'all_methods_identical':True,'improvement_resets_counter':True,'epoch_boundary_counter_replay':True,
            'best_selection_not_delayed':True}


## Manifest / initialization fairness and lightweight mechanism checks

No training or real dataset inference.

In [ ]:
# Actual initialization/manifest fairness for all three runs, before any training.
if (CROP_SCALE,TRAIN_CROP_FACTOR,EVAL_CROP_FACTOR)!=(1.5,1.5,1.5):raise RuntimeError('All phases require matched crop1.5')
if (DATA_SEED,GAIN_RANGE,PATIENCE_START_EPOCH,EARLY_STOP_PATIENCE,WARMUP_EPOCHS)!=(43,(.65,.90),15,3,1):raise RuntimeError('Frozen scale controls changed')
for filename,digest in MANIFEST_HASHES.items():
    if sha(Path(SCALE_MANIFEST_ROOT)/filename)!=digest or sha(out/'manifests'/filename)!=digest:raise RuntimeError('Frozen manifest changed since setup')
FAIR_CONFIGS={name:run_config(name,spec) for name,spec in RUNS.items()}
assert_fairness_contract(FAIR_CONFIGS,INITIAL_HASHES)
write_json(out/'fairness_checks.json',{'passed':True,'configs':FAIR_CONFIGS,'initial_state_sha256':INITIAL_HASHES,'manifest_sha256':MANIFEST_HASHES,
  'historical_C_eligibility':'notebook04 warmup_epochs0, epoch1 eligible','historical_P3_R7_eligibility':'notebook04/09 warmup_epoch1 excluded from best/patience'})

def mechanism_sanity_checks():
    class TinyPAD(nn.Module):
        def __init__(self):
            super().__init__();self.w=nn.Parameter(torch.tensor(1.));self.bn=nn.BatchNorm1d(3);self.calls=[];self.last_z=None
        def forward(self,x,aux=False):
            self.calls.append((self.training,torch.is_grad_enabled(),len(x)))
            a=self.bn(x.mean((2,3)));d=(a[:,0]-a[:,1])*self.w;z=torch.stack((d,-d),1);self.last_z=z
            return (z,d[:,None].expand(-1,11),d[:,None].expand(-1,5),d[:,None].expand(-1,40)) if aux else z
    global use_amp,select_worst_view
    old_amp=use_amp;original_selector=select_worst_view;use_amp=False
    try:
        with torch.random.fork_rng(devices=[]):
            x=torch.rand(4,3,80,80);y=torch.tensor([0,1,0,1]);labels=torch.zeros(4,dtype=torch.long);attrs=torch.zeros(4,40)
            masks=radial_masks(SCALED_CENTERS,P3_SF_SIGMA,device=torch.device('cpu'))
            p=TinyPAD().train();gain=torch.Generator().manual_seed(GAIN_SEED);state=gain.get_state().clone()
            original_selector(p,x,y,masks,gain)
            assert p.training and int(p.bn.num_batches_tracked)==0 and all(not train and not grad for train,grad,_ in p.calls)
            gain.set_state(state);first=original_selector(TinyPAD().train(),x,y,masks,gain)[1]
            gain.set_state(state);second=original_selector(TinyPAD().train(),x,y,masks,gain)[1]
            assert torch.equal(first,second)
            def forbidden(*args):raise AssertionError('C/warmup entered spectral selection')
            select_worst_view=forbidden
            for name,active in (('C_100K_crop15',True),('P3_SF_100K_crop15',False),('R7_SC_100K_crop15',False)):
                model=TinyPAD().train();loss,d,b,_=training_terms(model,x,y,labels,labels,attrs,RUNS[name],None,gain,active)
                assert len(model.calls)==1 and not b and d['spectral_ce']==0. and abs(loss.item()-d['clean_ce']-d['aux_loss'])<1e-6
            # Controlled detached selection metadata, no model/data fitting.
            def chosen_with_gate(harmful):
                def choose(*args):
                    cm=torch.full((4,),2.);wm=torch.where(harmful,torch.full((4,),1.),cm)
                    return x.roll(1,0),torch.tensor([0,1,2,0]),torch.full((4,),.75),torch.zeros(4),cm,wm
                return choose
            for gate in (torch.zeros(4,dtype=torch.bool),torch.tensor([True,False,True,False])):
                select_worst_view=chosen_with_gate(gate)
                model=TinyPAD().train();loss,d,b,_=training_terms(model,x,y,labels,labels,attrs,RUNS['R7_SC_100K_crop15'],masks,gain,True)
                assert d['harmful_count']==int(gate.sum()) and len(model.calls)==2
                assert abs(loss.item()-(.75*d['clean_ce']+.25*d['harmful_ce']+d['aux_loss']))<1e-6
                student_grad=torch.autograd.grad(loss,model.last_z,retain_graph=True)[0]
                if not gate.any():assert d['harmful_ce']==0. and torch.equal(student_grad,torch.zeros_like(student_grad))
                else:assert torch.equal(student_grad[~gate],torch.zeros_like(student_grad[~gate])) and student_grad[gate].abs().sum()>0
                loss.backward();assert model.w.grad is not None
            select_worst_view=original_selector
            model=TinyPAD().train();loss,d,_,_=training_terms(model,x,y,labels,labels,attrs,RUNS['P3_SF_100K_crop15'],masks,gain,True)
            assert abs(loss.item()-(.75*d['clean_ce']+.25*d['worst_ce']+d['aux_loss']))<1e-6
            assert len([c for c in model.calls if c[1]])==2 and int(model.bn.num_batches_tracked)==2
            saved=snapshot_rng(gain);draw=torch.rand(4,generator=gain);box=bbox_jitter({'bbox_xyxy':[10,10,50,50]})
            restore_rng(saved,gain)
            assert torch.equal(draw,torch.rand(4,generator=gain)) and box==bbox_jitter({'bbox_xyxy':[10,10,50,50]})
    finally:select_worst_view=original_selector;use_amp=old_amp
    return {'C_clean_only_epoch1_eligible':True,'P3_R7_clean_warmup_excluded':True,'BN_safe_detached_hard_selection':True,
      'P3_reference_objective':True,'R7_harmful_denominator_and_differentiable_zero':True,'R7_no_harmful_clean_coefficient_075':True,
      'one_selected_gradient_view':True,'gain_and_bbox_rng_restore':True}

IMPLEMENTATION_CHECKS=mechanism_sanity_checks()
IMPLEMENTATION_CHECKS['phase_aware_patience']=phase_patience_sanity_checks()
write_json(out/'implementation_sanity_checks.json',IMPLEMENTATION_CHECKS)
print('[Fairness/Sanity] Passed:',IMPLEMENTATION_CHECKS,flush=True)


## Reporting-only micro references and measured100K decisions

Missing runs/baseline remain pending. Secondary metrics never override the survival gate.

In [ ]:
# Supplied verified micro results: reporting only, never referenced by training.
MICRO_REFERENCES={'C_crop15':.830642,'P3_SF_crop15':.837525,'R7_SC_crop15':.832523}
MICRO_DELTAS={'P3_SF_100K_crop15':.006883,'R7_SC_100K_crop15':.001881}
STRONG_SURVIVAL_DELTA=.005
SCORE_SPLITS=('val','test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')
TABLE_COLUMNS=('selected_epoch','val_acer','val_auc','test_auc','test_acer','lcc_evaluation_auc','lcc_evaluation_hter',
 'lcc_combined_auc','delta_vs_C_100K','lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct','lcc_combined_detector_coverage')

def survival_status(delta):
    if not np.isfinite(delta):raise ValueError('Scale gate requires finite Combined-AUC delta')
    if delta>=STRONG_SURVIVAL_DELTA:return 'STRONG_SCALE_SURVIVAL'
    if delta>0:return 'WEAK_SCALE_SURVIVAL'
    return 'NO_SCALE_SURVIVAL'

def scale_decision(rows):
    measured={r['run_id']:r for r in rows if r['status']=='complete'}
    baseline=measured.get('C_100K_crop15')
    for row in rows:
        row['delta_vs_C_100K']=None;row['delta_vs_C_100K_pp']=None;row['retention_ratio']=None
        if row['status']!='complete':row['scale_survival_status']='PENDING'
        elif row['run_id']=='C_100K_crop15':row.update(delta_vs_C_100K=0.,delta_vs_C_100K_pp=0.,scale_survival_status='MATCHED_BASELINE')
        elif baseline is None:row['scale_survival_status']='PENDING_MATCHED_C_100K'
        else:
            delta=row['lcc_combined_auc']-baseline['lcc_combined_auc']
            row.update(delta_vs_C_100K=delta,delta_vs_C_100K_pp=100*delta,retention_ratio=delta/MICRO_DELTAS[row['run_id']],scale_survival_status=survival_status(delta))
    ranked=sorted(measured.values(),key=lambda r:-r['lcc_combined_auc'])
    complete=len(measured)==3;best=None;tied=[]
    if complete:
        tied=[r['run_id'] for r in ranked if round(r['lcc_combined_auc'],6)==round(ranked[0]['lcc_combined_auc'],6)]
        if len(tied)==1:best=tied[0]
    direct=(measured['P3_SF_100K_crop15']['lcc_combined_auc']-measured['R7_SC_100K_crop15']['lcc_combined_auc']) if {'P3_SF_100K_crop15','R7_SC_100K_crop15'}<=set(measured) else None
    if not complete:recommendation='Scale-survival interpretation pending all three matched100K runs. No new experiment is launched.'
    else:
        strong=[r['run_id'] for r in rows if r['scale_survival_status']=='STRONG_SCALE_SURVIVAL']
        weak=[r['run_id'] for r in rows if r['scale_survival_status']=='WEAK_SCALE_SURVIVAL']
        if len(strong)==2:recommendation='Both P3_SF and R7_SC are viable scaled candidates. Rank descriptively by Combined AUC; do not automatically combine their shared selected-worst mechanism.'
        elif strong and len(tied)>1:recommendation='Strong scale survival is observed, but the highest Combined AUC is tied at six-decimal reporting precision. Review manually; no superiority claim.'
        elif best=='P3_SF_100K_crop15' and best in strong:recommendation='Retain P3_SF as the current scaled candidate. One new final method-level variant may be worth testing as the third/final100K candidate; no local P3 weight tuning.'
        elif best=='R7_SC_100K_crop15' and best in strong:recommendation='Retain R7_SC as the current scaled candidate. Investigate a future final method based on selective/harmful spectral supervision; do not return to bandwidth/severity tuning.'
        elif weak:recommendation='The micro benefit survives directionally but is modest. Future development should introduce a different mechanism rather than small local weight/geometry tuning.'
        else:recommendation='Micro gains did not survive Train100K scale. Pivot method design before spending compute on an additional100K candidate.'
    return {'primary_endpoint':'lcc_combined_auc','gate_input':'measured Combined-AUC delta versus matched C_100K_crop15',
      'patience_start_epoch':PATIENCE_START_EPOCH,'early_stop_patience':EARLY_STOP_PATIENCE,
      'training_protocol_identical_to_older_scale100k':False,'strong_min_delta':STRONG_SURVIVAL_DELTA,'micro_references':MICRO_REFERENCES,'micro_deltas':MICRO_DELTAS,
      'ranking_complete':complete,'completed_ranking':[{'run_id':r['run_id'],'lcc_combined_auc':r['lcc_combined_auc']} for r in ranked],
      'best_100k_method':best,'top_ties_at_reporting_precision':tied if len(tied)>1 else [],
      'P3SF_vs_R7SC_100K_delta':direct,'P3SF_vs_R7SC_100K_delta_pp':100*direct if direct is not None else None,
      'rows':rows,'recommendation':recommendation,'engineering_screen_only':True}

def result_row(name,folder):
    row={'run_id':name,'status':'pending'}
    if not (folder/'completion.json').is_file():return row
    row.update(status='complete',selected_epoch=read_json(folder/'completion.json')['best_epoch'])
    for split in SCORE_SPLITS:
        for key,value in read_json(folder/'metrics'/(split+'_metrics.json')).items():row[split+'_'+key]=value
    return row

def value_text(value):
    return f'{value:.6f}' if isinstance(value,(int,float)) else 'pending'

def export_summary():
    rows=[result_row(name,out/'runs'/name) for name in RUNS];decision=scale_decision(rows)
    pd.DataFrame(rows).to_csv(out/'scale100k_C_P3SF_R7SC_comparison.csv',index=False)
    write_json(out/'scale100k_C_P3SF_R7SC_decision.json',decision)
    report=['# Matched crop1.5 scale-survival: C / P3_SF / R7_SC','',
      'Train100K/Val15K frozen manifests, DATA_SEED43/TRAIN_SEED100, identical official PAD-pretrained initialization. Shared clean inference architecture.',
      'New phase-aware patience: patience_start_epoch=15, early_stop_patience=3, earliest possible early-stop after epoch17; max20. Validation/best-checkpoint selection are not delayed.',
      'This100K experiment is not training-protocol-identical to older historical scale100K runs because of this new phase-aware patience rule.',
      'Historical notebook04 C is eligible from epoch1. P3/R7 epoch1 clean warmup is excluded from best/patience, preserving notebook04/09.',
      'P3_SF centers=[.0833333333,.25,.4166666667], sigma=.0555555556; R7_SC shares centers but sigma=.10. Gain Uniform(.65,.90).',
      'All Train/Val15K/full CelebA Test/LCC crops are1.5. Reload best; source-only threshold freeze precedes Test and LCC inference.',
      'Only Combined-AUC delta versus matched C_100K gates survival: >=.005 STRONG; >0 WEAK; otherwise NO. Secondary metrics and retention ratios cannot override it.',
      'Post-evaluation descriptive screening is not used to alter training, checkpoint selection, early stopping or thresholds.','',
      '| Run | '+' | '.join(TABLE_COLUMNS)+' | Scale Survival Status |','|---|'+'---:|'*len(TABLE_COLUMNS)+'---|']
    for row in rows:report.append('| '+row['run_id']+' | '+' | '.join(value_text(row.get(k)) for k in TABLE_COLUMNS)+' | '+row['scale_survival_status']+' |')
    report+=['','## Micro and100K deltas','',
      'Read-only verified micro references supplied in the request: C_crop15=.830642; P3_SF_crop15=.837525; R7_SC_crop15=.832523.',
      'Micro delta P3SF versus C=.006883 (+.6883 percentage points); R7SC versus C=.001881 (+.1881 percentage points).','',
      '| Run |100K delta versus C | Percentage points | Micro delta | Retention ratio |','|---|---:|---:|---:|---:|']
    for row in rows:
        if row['run_id'] in MICRO_DELTAS:report.append('| '+row['run_id']+' | '+' | '.join(value_text(v) for v in (row['delta_vs_C_100K'],row['delta_vs_C_100K_pp'],MICRO_DELTAS[row['run_id']],row['retention_ratio']))+' |')
    report+=['','R7_SC micro denominator .001881 is small; its retention ratio can be numerically large. Retention is contextual only, never gating/ranking/significance.',
      'P3SF versus R7SC100K Combined-AUC delta: '+value_text(decision['P3SF_vs_R7SC_100K_delta'])+' absolute; '+value_text(decision['P3SF_vs_R7SC_100K_delta_pp'])+' percentage points.',
      'Best100K method: '+str(decision['best_100k_method'] or 'pending all runs / top tie')+'. Completed Combined-AUC ranking: '+str(decision['completed_ranking'])+'. Descriptive only.','',
      '## Secondary metrics and coverage','']
    secondary={
      'val':('acer','auc','eer','locked_source_threshold','candidate_n','scored_n','detector_coverage'),
      'test':('auc','eer','apcer','bpcer','acer','binary_accuracy','candidate_n','scored_n','detector_coverage'),
      'lcc_training':('auc','eer','hter','candidate_n','scored_n','detector_coverage'),
      'lcc_development':('auc','eer','hter','candidate_n','scored_n','detector_coverage'),
      'lcc_evaluation':('auc','eer','hter','candidate_n','scored_n','detector_coverage'),
      'lcc_combined':('auc','eer','hter','tpr_at_fpr_1pct','candidate_n','scored_n','detector_coverage')}
    for split,keys in secondary.items():
        report+=['### '+split,'','| Run | '+' | '.join(keys)+' |','|---|'+'---:|'*len(keys)]
        for row in rows:report.append('| '+row['run_id']+' | '+' | '.join(value_text(row.get(split+'_'+key)) for key in keys)+' |')
        report.append('')
    report+=['## Training diagnostics','',
      'C clean/aux/total losses are in training_history.csv. P3/R7 spectral_diagnostics.csv retains active-epoch CE, margin/drop, flip, gain and selected-band counts/fractions.',
      'R7 harmful_count is a true epoch count; harmful_ce is pooled over harmful examples, harmful_ce_batch_mean retains the weighted batch diagnostic. The actual loss always uses each batch harmful denominator.',
      'Zero harmful samples produce differentiable zero; active clean coefficient stays.75. Harmful means margin reduction, even without misclassification. No additional training loss is added.','']
    for row in rows:
        if row['status']!='complete':continue
        folder=out/'runs'/row['run_id'];path=folder/('training_history.csv' if row['run_id']=='C_100K_crop15' else 'spectral_diagnostics.csv')
        if path.is_file():
            df=pd.read_csv(path)
            if not df.empty:
                last=df.iloc[-1];keys=('clean_ce','worst_ce','harmful_ce','harmful_count','harmful_fraction','aux_loss','train_total_loss','clean_aligned_margin','worst_margin','margin_drop','spectral_flip_rate','sampled_gain_mean','low_count','mid_count','high_count','low_fraction','mid_fraction','high_fraction')
                report.append('- '+row['run_id']+' last diagnostic epoch '+str(int(last.epoch))+': '+', '.join(k+'='+value_text(float(last[k])) for k in keys if k in df)+'.')
    report+=['','## Recommendation','',decision['recommendation'],'No automatic follow-up or claim of statistical superiority.','',
      '## Limitations','',
      '- One training seed; all scale-survival/ranking statements are descriptive.',
      '- LCC is a repeatedly used external development benchmark; this engineering screen is not an independent final-test claim.',
      '- Combined-AUC survival gate is predefined, not a significance test.',
      '- Full CelebA Test is measured only after source freeze and never used for training/checkpoint/threshold decisions.',
      '- Missing matched C or incomplete runs leave the corresponding deltas/decisions pending.',
      '- Multi-seed and additional independent validation remain future work; retention ratio alone does not establish robustness.']
    (out/'scale100k_C_P3SF_R7SC_report.md').write_text('\n'.join(report)+'\n')
    return rows

def export_run_zip(name):
    folder=out/'runs'/name;run_zip=out.parent/(name+'.zip')
    with zipfile.ZipFile(run_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(folder.rglob('*')):
            if file.is_file():archive.write(file,(Path('runs')/name/file.relative_to(folder)).as_posix())
        for filename in ('SCALE100K_C_P3SF_R7SC_CROP15_PROTOCOL_v1.md','manifest_sha256.json','initialization_fairness.json','fairness_checks.json','implementation_sanity_checks.json'):
            archive.write(out/filename,filename)
        for file in sorted((out/'manifests').glob('*.csv')):archive.write(file,(Path('manifests')/file.name).as_posix())
    display(FileLink(str(run_zip)))

def export_overall_zip():
    zip_path=out.parent/'17_scale100k_C_P3SF_R7SC_crop15.zip'
    print('[Export]',zip_path,flush=True)
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(out.rglob('*')):
            if file.is_file():archive.write(file,(Path(out.name)/file.relative_to(out)).as_posix())
    display(FileLink(str(zip_path)))

export_summary();export_overall_zip()


## Execute selected frozen100K runs on Kaggle

This cell starts expensive GPU training. Full Test/LCC inference occurs only after source freeze.

In [ ]:
selected=select_runs(RUN_FILTER)
print('[Runner] Selected runs:',selected,flush=True)
# Recheck fairness before starting any selected GPU training.
assert_fairness_contract({name:run_config(name,spec) for name,spec in RUNS.items()},INITIAL_HASHES)
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch256 ran out of GPU memory; stop instead of changing the recipe') from exc
    export_run_zip(name)  # Completed run is portable before reporting/starting another run.
    export_summary();export_overall_zip()
print('Finished selected scale-survival runs:',selected,flush=True)
for name in selected:display(FileLink(str(out.parent/(name+'.zip'))))
display(FileLink(str(out.parent/'17_scale100k_C_P3SF_R7SC_crop15.zip')))
